In [1]:
import sys
from pathlib import Path

root = Path.cwd()

if root.name in ["train", "notebooks"]:
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
import torch.nn as nn
import pandas as pd

from src.data.dataset import create_dataloaders
from src.models.cnn import CNNLanguageModel
from src.training.train import fit, evaluate

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


Cấu hình

In [2]:
CONTEXT_LENGTH = 64
BATCH_SIZE = 32

EPOCHS = 10
PATIENCE = 5

MODEL_NAME = "cnn"
EXPERIMENT_NAME = f"word_c{CONTEXT_LENGTH}"

Load Dataset

In [3]:
train_loader, valid_loader, test_loader, vocab_size = (
    create_dataloaders(
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE
    )
)

print("Vocabulary size:", vocab_size)
print("Train batches:", len(train_loader))

Vocabulary size: 10000
Train batches: 1042


Khởi tạo CNN

In [5]:
from src.models.cnn import CNNLanguageModel

model = CNNLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=256,
    hidden_dim=256,
    kernel_size=3,
    dropout=0.2,
    dilations=(1, 2, 4)
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", f"{total_params:,}")
print("Receptive field:", model.receptive_field)

Parameters: 5,720,592
Receptive field: 15


In [18]:
result = fit(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    model_name=MODEL_NAME,
    experiment_name=EXPERIMENT_NAME,
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
    root=root,
    epochs=EPOCHS,
    patience=PATIENCE
)


Epoch 1/10


Training:   0%|          | 0/1042 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/109 [00:00<?, ?it/s]

Train Loss: 5.2810 | Train PPL: 196.57
Valid Loss: 4.9243 | Valid PPL: 137.59
Đã lưu best model!

Epoch 2/10


Training:   0%|          | 0/1042 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/109 [00:00<?, ?it/s]

Train Loss: 4.4727 | Train PPL: 87.60
Valid Loss: 4.5084 | Valid PPL: 90.78
Đã lưu best model!

Epoch 10/10


Training:   0%|          | 0/1042 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/109 [00:00<?, ?it/s]

Train Loss: 4.4455 | Train PPL: 85.24
Valid Loss: 4.4958 | Valid PPL: 89.64
Đã lưu best model!

Training time: 3.68 phút


Test mô hình tốt nhất

In [19]:
model.load_state_dict(
    torch.load(
        result["checkpoint_path"],
        map_location=device,
        weights_only=True
    )
)

test_loss, test_ppl = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test PPL: {test_ppl:.2f}")

Evaluating:   0%|          | 0/123 [00:00<?, ?it/s]

Test Loss: 4.4253
Test PPL: 83.54


Lưu kết quả cuối cùng

In [20]:
summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "experiment": EXPERIMENT_NAME,
    "parameters": total_params,
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "best_valid_loss": result["best_valid_loss"],
    "best_valid_ppl": result["best_valid_ppl"],
    "test_loss": test_loss,
    "test_ppl": test_ppl,
    "epochs_trained": result["epochs_trained"],
    "training_time_seconds": result["training_time_seconds"]
}])

summary.to_csv(
    Path(result["results_dir"]) / "summary.csv",
    index=False
)

display(summary)

,model,experiment,parameters,context_length,batch_size,best_valid_loss,best_valid_ppl,test_loss,test_ppl,epochs_trained,training_time_seconds
0,cnn,word_c64,5720592,64,32,4.495764,89.636645,4.425313,83.538945,10,220.700684


In [21]:
import json

from src.training.inference import (
    predict_next_token,
    generate_text
)

with open(
    root / "data" / "processed" / "word2idx.json",
    encoding="utf-8"
) as f:
    word2idx = json.load(f)

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

model.load_state_dict(
    torch.load(
        result["checkpoint_path"],
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Đã nạp best CNN model!")

Đã nạp best CNN model!


In [22]:
prompt = "The president of the"

predictions = predict_next_token(
    model=model,
    text=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    top_k=5
)

print("Input:", prompt)
print("\nTop 5 predictions:")

for item in predictions:
    print(
        f"{item['token']:15s} "
        f"{item['probability'] * 100:.2f}%"
    )

Input: The president of the

Top 5 predictions:
Year            7.95%
United          2.71%
British         2.33%
American        1.55%
New             1.30%


In [25]:
prompt = "The history of the"

generated = generate_text(
    model=model,
    prompt=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    max_new_tokens=100,
    temperature=0.8,
    top_k=20
)

print("Prompt:", prompt)
print("\nGenerated text:")
print(generated)

Prompt: The history of the

Generated text:
The history of the first @ - @ class , and a local success , and in which he was a part of the club . After her third album , the band ' s favorite tour .
